In [1]:
import os
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_google_genai import ChatGoogleGenerativeAI

C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
API_KEY = 'API_KEY'

In [3]:
llm = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    google_api_key = API_KEY,
    temperature = 0.2
)

In [4]:
class HealthcareState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    patient_input: str
    patient_history: str
 
    # Stage 1 outputs
    extracted_symptoms: str
    severity_score: str 
    emergency_flags: str
    distress_level: str
    medical_entities: str
 
    # Stage 2 outputs (knowledge retrieval)
    medical_guidelines: str
    drug_interaction_data: str
    specialist_recommendations: str
    similar_cases: str
 
    # Stage 3 output
    routed_to: str
 
    # Stage 4 outputs
    triage_summary: str
    suggested_specialist: str
    follow_up_instructions: str
    diagnostic_tests: str
    emergency_notification: str
    final_care_plan: str

In [5]:
def symptom_extraction_node(state: HealthcareState) -> dict:
    """Extract symptoms from patient input."""
    prompt = f"""You are a clinical NLP system.
Patient complaint: "{state['patient_input']}"
Patient history: "{state.get('patient_history', 'None provided')}"
 
Respond in EXACTLY this format:
extracted_symptoms: <comma-separated list of symptoms mentioned>
medical_entities: <comma-separated list of body parts / conditions / medications mentioned>"""
 
    response = llm.invoke([HumanMessage(content=prompt)])
    lines = {k.strip(): v.strip() for k, v in
             (line.split(":", 1) for line in response.content.strip().splitlines() if ":" in line)}
 
    return {
        "extracted_symptoms": lines.get("extracted_symptoms", state["patient_input"]),
        "medical_entities": lines.get("medical_entities", ""),
        "messages": [AIMessage(content=f"[Symptom Extraction] {lines}")],
    }
 
 
def risk_scoring_node(state: HealthcareState) -> dict:
    """Score clinical severity."""
    prompt = f"""You are a clinical risk scoring system.
Patient complaint: "{state['patient_input']}"
Patient history: "{state.get('patient_history', 'None provided')}"
 
Respond in EXACTLY this format:
severity_score: <one of: low | medium | high | emergency>
emergency_flags: <brief list of red-flag findings, or 'none'>
 
Rules:
- emergency: chest pain + shortness of breath, sudden vision loss, stroke signs, unconsciousness
- high: severe pain, high fever (>103°F), uncontrolled bleeding
- medium: persistent fever <103°F, moderate pain, non-urgent infections
- low: minor rash, mild cold, routine follow-up"""
 
    response = llm.invoke([HumanMessage(content=prompt)])
    lines = {k.strip(): v.strip() for k, v in
             (line.split(":", 1) for line in response.content.strip().splitlines() if ":" in line)}
 
    return {
        "severity_score": lines.get("severity_score", "medium"),
        "emergency_flags": lines.get("emergency_flags", "none"),
        "messages": [AIMessage(content=f"[Risk Scoring] severity={lines.get('severity_score')}, flags={lines.get('emergency_flags')}")],
    }
 
 
def distress_analysis_node(state: HealthcareState) -> dict:
    """Analyze patient distress level from language."""
    prompt = f"""Analyze the emotional distress level in this patient message.
Message: "{state['patient_input']}"
 
Respond in EXACTLY this format:
distress_level: <one of: calm | distressed | critical_distress>"""
 
    response = llm.invoke([HumanMessage(content=prompt)])
    lines = {k.strip(): v.strip() for k, v in
             (line.split(":", 1) for line in response.content.strip().splitlines() if ":" in line)}
 
    return {
        "distress_level": lines.get("distress_level", "calm"),
        "messages": [AIMessage(content=f"[Distress Analysis] distress_level={lines.get('distress_level')}")],
    }

In [6]:
def knowledge_retrieval_node(state: HealthcareState) -> dict:
    symptoms = state.get("extracted_symptoms", state["patient_input"])
    entities = state.get("medical_entities", "")
    history = state.get("patient_history", "")
    severity = state.get("severity_score", "medium")

    prompt = f"""You are a medical knowledge system.
Patient symptoms: {symptoms}
Medical entities: {entities}
Patient history: {history}
Severity: {severity}

Respond in EXACTLY this format (no extra text):
medical_guidelines: <2-3 relevant clinical guidelines in one line>
drug_interactions: <any drug interactions or 'none'>
specialist_recommendations: <1-2 specialist departments with one word justification>
similar_cases: <one similar historical case in one line>"""

    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    parsed = {k.strip(): v.strip() for k, v in (line.split(":", 1) for line in response.splitlines() if ":" in line)}

    return {
        "medical_guidelines": parsed.get("medical_guidelines", ""),
        "drug_interaction_data": parsed.get("drug_interactions", "none"),
        "specialist_recommendations": parsed.get("specialist_recommendations", ""),
        "similar_cases": parsed.get("similar_cases", ""),
        "messages": [AIMessage(content=f"[Knowledge Retrieval] {parsed}")],
    }

In [7]:
 
def triage_routing_node(state: HealthcareState) -> dict:
    """Decide routing based on severity and symptoms."""
    severity = state.get("severity_score", "medium")
    symptoms = state.get("extracted_symptoms", "").lower()
    entities = state.get("medical_entities", "").lower()
    emergency_flags = state.get("emergency_flags", "none").lower()
 
    if severity == "emergency" or emergency_flags != "none":
        routed_to = "emergency_escalation_agent"
    elif any(kw in symptoms + entities for kw in ["chest", "cardiac", "heart", "palpitation", "ecg"]):
        routed_to = "cardiology_agent"
    elif any(kw in symptoms + entities for kw in ["dizzy", "dizziness", "blurred vision", "neuro", "headache", "migraine", "brain"]):
        routed_to = "neuro_specialist_agent"
    elif any(kw in symptoms + entities for kw in ["rash", "medication", "drug", "allergy", "reaction"]):
        routed_to = "pharmacy_review_agent"
    else:
        routed_to = "general_physician_agent"
 
    return {
        "routed_to": routed_to,
        "messages": [AIMessage(content=f"[Triage Router] Routing to → {routed_to}")],
    }
 
 
def triage_router_edge(state: HealthcareState) -> Literal[
    "emergency_escalation_agent", "cardiology_agent",
    "neuro_specialist_agent", "pharmacy_review_agent", "general_physician_agent"
]:
    return state["routed_to"]

In [8]:
 
def _specialist_assessment(role: str, context: dict) -> str:
    prompt = f"""You are the {role} reviewing a triage case.
Context:
{chr(10).join(f'  {k}: {v}' for k, v in context.items())}
 
Provide your clinical assessment and immediate action plan (3-4 sentences)."""
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()
 
 
def emergency_escalation_agent_node(state: HealthcareState) -> dict:
    assessment = _specialist_assessment("Emergency Medicine Specialist", {
        "patient_input": state["patient_input"],
        "severity_score": state.get("severity_score"),
        "emergency_flags": state.get("emergency_flags"),
        "medical_guidelines": state.get("medical_guidelines"),
    })
    return {"routed_to": "emergency_escalation_agent",
            "messages": [AIMessage(content=f"[Emergency Agent] {assessment}")]}
 
 
def cardiology_agent_node(state: HealthcareState) -> dict:
    assessment = _specialist_assessment("Cardiologist", {
        "symptoms": state.get("extracted_symptoms"),
        "entities": state.get("medical_entities"),
        "severity": state.get("severity_score"),
        "guidelines": state.get("medical_guidelines"),
    })
    return {"routed_to": "cardiology_agent",
            "messages": [AIMessage(content=f"[Cardiology Agent] {assessment}")]}
 
 
def neuro_specialist_agent_node(state: HealthcareState) -> dict:
    assessment = _specialist_assessment("Neurologist", {
        "symptoms": state.get("extracted_symptoms"),
        "severity": state.get("severity_score"),
        "similar_cases": state.get("similar_cases"),
    })
    return {"routed_to": "neuro_specialist_agent",
            "messages": [AIMessage(content=f"[Neuro Agent] {assessment}")]}
 
 
def pharmacy_review_agent_node(state: HealthcareState) -> dict:
    assessment = _specialist_assessment("Clinical Pharmacist", {
        "symptoms": state.get("extracted_symptoms"),
        "drug_interactions": state.get("drug_interaction_data"),
        "entities": state.get("medical_entities"),
    })
    return {"routed_to": "pharmacy_review_agent",
            "messages": [AIMessage(content=f"[Pharmacy Agent] {assessment}")]}
 
 
def general_physician_agent_node(state: HealthcareState) -> dict:
    assessment = _specialist_assessment("General Physician", {
        "symptoms": state.get("extracted_symptoms"),
        "severity": state.get("severity_score"),
        "guidelines": state.get("medical_guidelines"),
    })
    return {"routed_to": "general_physician_agent",
            "messages": [AIMessage(content=f"[GP Agent] {assessment}")]}

In [9]:

def triage_summary_node(state: HealthcareState) -> dict:
    prompt = f"""Generate a concise clinical triage summary.
Patient complaint: "{state['patient_input']}"
Symptoms extracted: {state.get('extracted_symptoms')}
Severity: {state.get('severity_score')}
Emergency flags: {state.get('emergency_flags')}
Distress level: {state.get('distress_level')}
Routed to: {state.get('routed_to')}
 
Write a 3-4 sentence triage summary suitable for a handoff note."""
 
    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return {
        "triage_summary": response,
        "messages": [AIMessage(content=f"[Triage Summary]\n{response}")],
    }
 
 
def suggested_specialist_node(state: HealthcareState) -> dict:
    specialist_map = {
        "emergency_escalation_agent": "Emergency Medicine / ICU",
        "cardiology_agent": "Cardiology",
        "neuro_specialist_agent": "Neurology",
        "pharmacy_review_agent": "Clinical Pharmacology",
        "general_physician_agent": "General Medicine",
    }
    specialist = specialist_map.get(state.get("routed_to", ""), "General Medicine")
    return {
        "suggested_specialist": specialist,
        "messages": [AIMessage(content=f"[Specialist Suggestion] Refer to: {specialist}")],
    }
 
 
def follow_up_instructions_node(state: HealthcareState) -> dict:
    prompt = f"""Create patient follow-up instructions.
Symptoms: {state.get('extracted_symptoms')}
Severity: {state.get('severity_score')}
Specialist: {state.get('routed_to')}
Medical guidelines: {state.get('medical_guidelines')}
 
Write 3-5 clear, numbered follow-up instructions for the patient."""
 
    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return {
        "follow_up_instructions": response,
        "messages": [AIMessage(content=f"[Follow-up Instructions]\n{response}")],
    }
 
 
def diagnostic_tests_node(state: HealthcareState) -> dict:
    prompt = f"""Recommend diagnostic tests for this patient.
Symptoms: {state.get('extracted_symptoms')}
Medical entities: {state.get('medical_entities')}
Severity: {state.get('severity_score')}
Routed specialist: {state.get('routed_to')}
 
List 3-5 recommended diagnostic tests with brief justification."""
 
    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return {
        "diagnostic_tests": response,
        "messages": [AIMessage(content=f"[Diagnostic Tests]\n{response}")],
    }
 
 
def emergency_notification_node(state: HealthcareState) -> dict:
    """Only fires a real alert for emergency cases."""
    severity = state.get("severity_score", "low")
    flags = state.get("emergency_flags", "none")
 
    if severity == "emergency" or flags.lower() != "none":
        notification = (
            f"EMERGENCY ALERT: Patient presenting with '{state['patient_input']}'. "
            f"Flags: {flags}. Immediate attention required at Emergency Desk. "
            f"Routed to: {state.get('routed_to')}."
        )
    else:
        notification = "No emergency notification required for this case."
 
    return {
        "emergency_notification": notification,
        "messages": [AIMessage(content=f"[Emergency Notification] {notification}")],
    }

In [10]:
 
def final_care_plan_node(state: HealthcareState) -> dict:
    plan = f"""
========== CLINICAL TRIAGE & CARE COORDINATION REPORT ==========
Patient Complaint   : {state['patient_input']}
Extracted Symptoms  : {state.get('extracted_symptoms')}
Medical Entities    : {state.get('medical_entities')}
Severity Score      : {state.get('severity_score')}
Emergency Flags     : {state.get('emergency_flags')}
Distress Level      : {state.get('distress_level')}
Routed To           : {state.get('routed_to')}
Suggested Specialist: {state.get('suggested_specialist')}
 
--- Emergency Notification ---
{state.get('emergency_notification')}
 
--- Triage Summary ---
{state.get('triage_summary')}
 
--- Diagnostic Tests ---
{state.get('diagnostic_tests')}
 
--- Follow-up Instructions ---
{state.get('follow_up_instructions')}
 
--- Drug Interaction Data ---
{state.get('drug_interaction_data')}
 
--- Similar Historical Cases ---
{state.get('similar_cases')}
================================================================="""
 
    return {
        "final_care_plan": plan,
        "messages": [AIMessage(content=plan)],
    }

In [11]:
def build_healthcare_graph() -> StateGraph:
    graph = StateGraph(HealthcareState)

    # Stage 1 — parallel
    graph.add_node("symptom_extraction", symptom_extraction_node)
    graph.add_node("risk_scoring", risk_scoring_node)
    graph.add_node("distress_analysis", distress_analysis_node)

    # Stage 2 — single combined retrieval node
    graph.add_node("knowledge_retrieval", knowledge_retrieval_node)

    # Stage 3 — routing
    graph.add_node("triage_routing", triage_routing_node)
    graph.add_node("emergency_escalation_agent", emergency_escalation_agent_node)
    graph.add_node("cardiology_agent", cardiology_agent_node)
    graph.add_node("neuro_specialist_agent", neuro_specialist_agent_node)
    graph.add_node("pharmacy_review_agent", pharmacy_review_agent_node)
    graph.add_node("general_physician_agent", general_physician_agent_node)

    # Stage 4 — parallel care coordination
    graph.add_node("triage_summary", triage_summary_node)
    graph.add_node("suggested_specialist", suggested_specialist_node)
    graph.add_node("follow_up_instructions", follow_up_instructions_node)
    graph.add_node("diagnostic_tests", diagnostic_tests_node)
    graph.add_node("emergency_notification", emergency_notification_node)
    graph.add_node("final_care_plan", final_care_plan_node)

    # START → Stage 1 (parallel)
    for s1 in ["symptom_extraction", "risk_scoring", "distress_analysis"]:
        graph.add_edge(START, s1)

    # Stage 1 → Stage 2 (fan-in to single retrieval node)
    for s1 in ["symptom_extraction", "risk_scoring", "distress_analysis"]:
        graph.add_edge(s1, "knowledge_retrieval")

    # Stage 2 → Triage Routing
    graph.add_edge("knowledge_retrieval", "triage_routing")

    # Conditional routing → Specialist agents
    graph.add_conditional_edges(
        "triage_routing",
        triage_router_edge,
        {
            "emergency_escalation_agent": "emergency_escalation_agent",
            "cardiology_agent": "cardiology_agent",
            "neuro_specialist_agent": "neuro_specialist_agent",
            "pharmacy_review_agent": "pharmacy_review_agent",
            "general_physician_agent": "general_physician_agent",
        },
    )

    # Specialist agents → Stage 4 (parallel)
    specialist_agents = [
        "emergency_escalation_agent", "cardiology_agent",
        "neuro_specialist_agent", "pharmacy_review_agent", "general_physician_agent",
    ]
    stage4_nodes = ["triage_summary", "suggested_specialist", "follow_up_instructions",
                    "diagnostic_tests", "emergency_notification"]

    for agent in specialist_agents:
        for s4 in stage4_nodes:
            graph.add_edge(agent, s4)

    # Stage 4 → Final care plan
    for s4 in stage4_nodes:
        graph.add_edge(s4, "final_care_plan")

    graph.add_edge("final_care_plan", END)

    return graph.compile()

In [13]:
 
def run_healthcare_system(patient_input: str, patient_history: str = ""):
    print(f"PATIENT INPUT: {patient_input}")
 
    app = build_healthcare_graph()
    initial_state: HealthcareState = {
        "messages": [HumanMessage(content=patient_input)],
        "patient_input": patient_input,
        "patient_history": patient_history,
        "extracted_symptoms": "",
        "severity_score": "",
        "emergency_flags": "",
        "distress_level": "",
        "medical_entities": "",
        "medical_guidelines": "",
        "drug_interaction_data": "",
        "specialist_recommendations": "",
        "similar_cases": "",
        "routed_to": "",
        "triage_summary": "",
        "suggested_specialist": "",
        "follow_up_instructions": "",
        "diagnostic_tests": "",
        "emergency_notification": "",
        "final_care_plan": "",
    }
 
    result = app.invoke(initial_state)
    return result["final_care_plan"]
 
 
if __name__ == "__main__":
    cases = [
        ("Chest pain and shortness of breath.", "Hypertension, on aspirin 75mg")
    ]
    for patient_input, history in cases:
        print(run_healthcare_system(patient_input, history))
        print("\n")
 

PATIENT INPUT: Chest pain and shortness of breath.

========== CLINICAL TRIAGE & CARE COORDINATION REPORT ==========
Patient Complaint   : Chest pain and shortness of breath.
Extracted Symptoms  : Chest pain, shortness of breath
Medical Entities    : Hypertension, Aspirin 75mg
Severity Score      : emergency
Emergency Flags     : Chest pain, Shortness of breath
Distress Level      : critical_distress
Routed To           : emergency_escalation_agent
Suggested Specialist: Emergency Medicine / ICU

--- Emergency Notification ---
EMERGENCY ALERT: Patient presenting with 'Chest pain and shortness of breath.'. Flags: Chest pain, Shortness of breath. Immediate attention required at Emergency Desk. Routed to: emergency_escalation_agent.

--- Triage Summary ---
Patient presents with acute chest pain and shortness of breath, exhibiting critical distress. These symptoms are identified as emergency flags, classifying the case with an emergency severity. The patient has been immediately routed to t